# Day2 - Bronze: MovieLens Tables

**Objetivo:** Criar as 4 tabelas Bronze do dataset MovieLens (`links`, `movies`, `ratings`, `tags`) a partir dos arquivos CSV na camada Landing.

| Item | Valor |
|---|---|
| **Camada** | Bronze |
| **Entrada** | `/Volumes/{catalog}/landing/files/movielens/*.csv` |
| **Saída** | `{catalog}.bronze.links`, `.movies`, `.ratings`, `.tags` |
| **Ordem no pipeline** | Após upload dos CSVs para a Landing |

In [0]:
dbutils.widgets.text("catalog", "capgemini_academy", "Nome do catálogo no Unity Catalog")
catalog = dbutils.widgets.get("catalog")

## Tabela `bronze.links`

Mapeia MovieLens IDs para IMDb e TMDb. `CREATE OR REPLACE TABLE` + `INSERT` garante idempotência.

In [0]:
%sql
CREATE OR REPLACE TABLE `${catalog}`.`bronze`.`links` (
  movieId INT COMMENT 'Movie ID from MovieLens',
  imdbId INT COMMENT 'IMDb ID',
  tmdbId INT COMMENT 'The Movie Database (TMDb) ID',
  updated TIMESTAMP COMMENT 'Timestamp of last update'
)
USING DELTA
COMMENT 'MovieLens links table - maps MovieLens IDs to external movie databases';

INSERT INTO `${catalog}`.`bronze`.`links`
SELECT movieId, imdbId, tmdbId, from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo') AS updated
FROM read_files(
  '/Volumes/${catalog}/landing/files/movielens/links.csv',
  format => 'csv',
  header => true
);

## Tabela `bronze.movies`

Contém títulos e gêneros (pipe-separated) dos filmes.

In [0]:
%sql
CREATE OR REPLACE TABLE `${catalog}`.`bronze`.`movies` (
  movieId INT COMMENT 'Movie ID from MovieLens',
  title STRING COMMENT 'Movie title',
  genres STRING COMMENT 'Movie genres (pipe-separated)',
  updated TIMESTAMP COMMENT 'Timestamp of last update'
)
USING DELTA
COMMENT 'MovieLens movies table - contains movie titles and genres';

INSERT INTO `${catalog}`.`bronze`.`movies`
SELECT movieId, title, genres, from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo') AS updated
FROM read_files(
  '/Volumes/${catalog}/landing/files/movielens/movies.csv',
  format => 'csv',
  header => true
);

## Tabela `bronze.ratings`

Avaliações dos usuários (1–5). `rating` é `INT` — valores decimais são truncados.

In [0]:
%sql
CREATE OR REPLACE TABLE `${catalog}`.`bronze`.`ratings` (
  userId INT COMMENT 'User ID from MovieLens',
  movieId INT COMMENT 'Movie ID from MovieLens',
  rating INT COMMENT 'Movie rating (1.0 - 5.0)',
  timestamp TIMESTAMP COMMENT 'Timestamp of rating',
  updated TIMESTAMP COMMENT 'Timestamp of last update'
)
USING DELTA
COMMENT 'MovieLens ratings table - contains user ratings for movies';

INSERT INTO `${catalog}`.`bronze`.`ratings`
SELECT userId, movieId, rating, from_unixtime(timestamp) AS timestamp, from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo') AS updated
FROM read_files(
  '/Volumes/${catalog}/landing/files/movielens/ratings.csv',
  format => 'csv',
  header => true
);

## Tabela `bronze.tags`

Tags atribuídas pelos usuários aos filmes.

In [0]:
%sql
CREATE OR REPLACE TABLE `${catalog}`.`bronze`.`tags` (
  userId INT COMMENT 'User ID from MovieLens',
  movieId INT COMMENT 'Movie ID from MovieLens',
  tag STRING COMMENT 'Movie tag',
  timestamp TIMESTAMP COMMENT 'Timestamp of rating',
  updated TIMESTAMP COMMENT 'Timestamp of last update'
)
USING DELTA
COMMENT 'MovieLens tags table - contains user tags for movies';

INSERT INTO `${catalog}`.`bronze`.`tags`
SELECT userId, movieId, tag, from_unixtime(timestamp) AS timestamp, from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo') AS updated
FROM read_files(
  '/Volumes/${catalog}/landing/files/movielens/tags.csv',
  format => 'csv',
  header => true
);